# Week 5 Day 4 — Advanced Feature Engineering

## What I built:
- Skewness analysis — 23 of 43 numeric features highly skewed
- Log transform skewed features — mean skewness 5.43 → 2.47
- Target encoding — Neighborhood → mean SalePrice (corr 0.74)
- Interaction features — TotalSF×Qual (corr 0.83), GrLiv×Qual (0.83)
- Systematic experiment — tested each addition separately

## Experiment results:
Baseline                 : 0.1279
+ Neighborhood encoding  : 0.1272 ← helped CV
+ Interaction features   : 0.1293 ← hurt

## Key learning:
More features ≠ better model
LightGBM handles skewed features internally
Tree models already capture interactions
Test each feature addition empirically — never assume

## TotalSF_Age explanation:
TotalSF × (1/(HouseAge+1))
→ large new house scores highest
→ +1 prevents division by zero (Laplace smoothing)
→ captures recency + size combined

## Kaggle scores:
LightGBM tuned (Day 3)      : 0.12603 ← still best
LightGBM + neighborhood enc : 0.12612 ← slightly worse

# Find and fix skewed numeric features:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, KFold
import warnings
warnings.filterwarnings('ignore')

# load data
train = pd.read_csv('house_prices_train.csv')
test  = pd.read_csv('house_prices_test.csv')

def engineer_features(df):
    df = df.copy()
    df['TotalSF']     = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']   = (df['FullBath'] + 0.5*df['HalfBath'] +
                         df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['HouseAge']    = df['YrSold'] - df['YearBuilt']
    df['RemodAge']    = df['YrSold'] - df['YearRemodAdd']
    df['HasPool']     = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']   = (df['GarageArea'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

# find skewed numeric features
num_cols = train_fe.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['SalePrice', 'Id']]

skewness = train_fe[num_cols].skew().sort_values(ascending=False)
skewed_cols = skewness[skewness > 0.75].index.tolist()

print(f"Total numeric features  : {len(num_cols)}")
print(f"Highly skewed (>0.75)   : {len(skewed_cols)}")
print("\nTop 10 most skewed:")
print(skewness.head(10))

Total numeric features  : 43
Highly skewed (>0.75)   : 23

Top 10 most skewed:
MiscVal         24.476794
PoolArea        14.828374
HasPool         14.352680
LotArea         12.207688
3SsnPorch       10.304342
LowQualFinSF     9.011341
KitchenAbvGr     4.488397
BsmtFinSF2       4.255261
ScreenPorch      4.122214
BsmtHalfBath     4.103403
dtype: float64


# Log transform skewed features:

In [2]:
from scipy.stats import skew

def fix_skewness(train_df, test_df, threshold=0.75):
    """Log transform all numeric features with skewness above threshold."""
    train_df = train_df.copy()
    test_df  = test_df.copy()
    
    num_cols = train_df.select_dtypes(include=['int64','float64']).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['SalePrice', 'Id']]
    
    fixed = []
    for col in num_cols:
        if train_df[col].skew() > threshold:
            # log1p handles zeros safely
            train_df[col] = np.log1p(train_df[col])
            test_df[col]  = np.log1p(test_df[col])
            fixed.append(col)
    
    print(f"Log transformed {len(fixed)} skewed features")
    return train_df, test_df

train_fixed, test_fixed = fix_skewness(train_fe, test_fe)

# verify skewness reduced
skewness_after = train_fixed[skewed_cols].skew().sort_values(ascending=False)
print("\nTop 10 skewness AFTER fixing:")
print(skewness_after.head(10))
print(f"\nMean skewness before: {skewness[skewed_cols].mean():.4f}")
print(f"Mean skewness after : {skewness_after.mean():.4f}")

Log transformed 23 skewed features

Top 10 skewness AFTER fixing:
PoolArea         14.363102
HasPool          14.352680
3SsnPorch         7.734975
LowQualFinSF      7.460317
MiscVal           5.170704
BsmtHalfBath      3.933064
KitchenAbvGr      3.869414
ScreenPorch       3.150409
BsmtFinSF2        2.523694
EnclosedPorch     2.112275
dtype: float64

Mean skewness before: 5.4282
Mean skewness after : 2.4667


# Target encoding for Neighborhood:

In [3]:
# target encoding — replace neighborhood name with mean SalePrice
neighborhood_mean = train_fixed.groupby('Neighborhood')['SalePrice'].mean()

print("Mean SalePrice by Neighborhood (top 10):")
print(neighborhood_mean.sort_values(ascending=False).head(10))

# apply to train and test
train_fixed['Neighborhood_encoded'] = train_fixed['Neighborhood'].map(neighborhood_mean)
test_fixed['Neighborhood_encoded']  = test_fixed['Neighborhood'].map(neighborhood_mean)

# handle any unseen neighborhoods in test
test_fixed['Neighborhood_encoded'].fillna(neighborhood_mean.mean(), inplace=True)

print(f"\nNeighborhood_encoded correlation with SalePrice: "
      f"{train_fixed['Neighborhood_encoded'].corr(train_fixed['SalePrice']):.4f}")
print(f"Neighborhood (original) in cat_cols — will be dropped from pipeline")

Mean SalePrice by Neighborhood (top 10):
Neighborhood
NoRidge    335295.317073
NridgHt    316270.623377
StoneBr    310499.000000
Timber     242247.447368
Veenker    238772.727273
Somerst    225379.837209
ClearCr    212565.428571
Crawfor    210624.725490
CollgCr    197965.773333
Blmngtn    194870.882353
Name: SalePrice, dtype: float64

Neighborhood_encoded correlation with SalePrice: 0.7386
Neighborhood (original) in cat_cols — will be dropped from pipeline


# Interaction features:

In [4]:
# interaction features — multiply top correlated features
train_fixed['TotalSF_Qual']    = train_fixed['TotalSF'] * train_fixed['OverallQual']
train_fixed['GrLiv_Qual']      = train_fixed['GrLivArea'] * train_fixed['OverallQual']
train_fixed['Garage_Qual']     = train_fixed['GarageArea'] * train_fixed['OverallQual']
train_fixed['TotalSF_Age']     = train_fixed['TotalSF'] * (1 / (train_fixed['HouseAge'] + 1))

test_fixed['TotalSF_Qual']     = test_fixed['TotalSF'] * test_fixed['OverallQual']
test_fixed['GrLiv_Qual']       = test_fixed['GrLivArea'] * test_fixed['OverallQual']
test_fixed['Garage_Qual']      = test_fixed['GarageArea'] * test_fixed['OverallQual']
test_fixed['TotalSF_Age']      = test_fixed['TotalSF'] * (1 / (test_fixed['HouseAge'] + 1))

# check correlations
new_features = ['TotalSF_Qual', 'GrLiv_Qual', 'Garage_Qual', 'TotalSF_Age']
print("Correlation of interaction features with SalePrice:")
for feat in new_features:
    corr = train_fixed[feat].corr(train_fixed['SalePrice'])
    print(f"  {feat:20} : {corr:.4f}")

Correlation of interaction features with SalePrice:
  TotalSF_Qual         : 0.8309
  GrLiv_Qual           : 0.8302
  Garage_Qual          : 0.7900
  TotalSF_Age          : 0.4555


# Train model with all new features:

In [5]:
# update feature lists with new columns
X_new = train_fixed.drop(columns=['SalePrice', 'Id', 'Neighborhood'])
y_new = np.log1p(train_fixed['SalePrice'])

num_cols_new = X_new.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols_new = X_new.select_dtypes(include=['object']).columns.tolist()

# pipeline
num_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])
preprocessor_new = ColumnTransformer(transformers=[
    ('num', num_tr, num_cols_new),
    ('cat', cat_tr, cat_cols_new)
])

lgb_pipe_new = Pipeline(steps=[
    ('preprocessor', preprocessor_new),
    ('model', lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.005,
        max_depth=5,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_samples=20,
        reg_alpha=0.5,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1
    ))
])

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_new = cross_val_score(lgb_pipe_new, X_new, y_new,
                              cv=kfold,
                              scoring='neg_root_mean_squared_error')

print("=" * 55)
print("   LIGHTGBM WITH ADVANCED FEATURES")
print("=" * 55)
print(f"  Previous RMSE : 0.1274")
print(f"  New RMSE      : {-scores_new.mean():.4f} ± {scores_new.std():.4f}")
print(f"  Improvement   : {0.1274 - (-scores_new.mean()):+.4f}")
print("=" * 55)

   LIGHTGBM WITH ADVANCED FEATURES
  Previous RMSE : 0.1274
  New RMSE      : 0.1289 ± 0.0134
  Improvement   : -0.0015


# Test features one by one to find what helps:

In [6]:
# test each addition separately
experiments = {
    'Baseline (no new features)'     : train_fe,
    'With skewness fix'              : train_fixed,
    'With neighborhood encoding'     : None,  # we'll build below
    'With interaction features only' : None,
}

# experiment 1 — baseline
X_base = train_fe.drop(columns=['SalePrice', 'Id'])
y_base = np.log1p(train_fe['SalePrice'])
num_base = X_base.select_dtypes(include=['int64','float64']).columns.tolist()
cat_base = X_base.select_dtypes(include=['object']).columns.tolist()

# experiment 2 — only neighborhood encoding (no skewness fix)
train_nb = train_fe.copy()
train_nb['Neighborhood_encoded'] = train_nb['Neighborhood'].map(neighborhood_mean)
train_nb = train_nb.drop(columns=['Neighborhood'])
X_nb = train_nb.drop(columns=['SalePrice', 'Id'])
y_nb = np.log1p(train_nb['SalePrice'])
num_nb = X_nb.select_dtypes(include=['int64','float64']).columns.tolist()
cat_nb = X_nb.select_dtypes(include=['object']).columns.tolist()

# experiment 3 — only interaction features (no skewness fix)
train_int = train_fe.copy()
train_int['TotalSF_Qual'] = train_int['TotalSF'] * train_int['OverallQual']
train_int['GrLiv_Qual']   = train_int['GrLivArea'] * train_int['OverallQual']
X_int = train_int.drop(columns=['SalePrice', 'Id'])
y_int = np.log1p(train_int['SalePrice'])
num_int = X_int.select_dtypes(include=['int64','float64']).columns.tolist()
cat_int = X_int.select_dtypes(include=['object']).columns.tolist()

def get_lgb_score(X, y, num_c, cat_c):
    num_t = Pipeline(steps=[('i', SimpleImputer(strategy='median')),('s', StandardScaler())])
    cat_t = Pipeline(steps=[('i', SimpleImputer(strategy='most_frequent')),
                             ('e', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
    pre = ColumnTransformer(transformers=[('num', num_t, num_c), ('cat', cat_t, cat_c)])
    pipe = Pipeline(steps=[('preprocessor', pre),
                            ('model', lgb.LGBMRegressor(
                                n_estimators=2000, learning_rate=0.005,
                                num_leaves=63, subsample=0.8,
                                colsample_bytree=0.5, min_child_samples=20,
                                reg_alpha=0.5, reg_lambda=0.1,
                                random_state=42, verbose=-1))])
    scores = cross_val_score(pipe, X, y, cv=kfold,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

print("=" * 55)
print("   FEATURE EXPERIMENT RESULTS")
print("=" * 55)
print(f"  Baseline                 : {get_lgb_score(X_base, y_base, num_base, cat_base):.4f}")
print(f"  + Neighborhood encoding  : {get_lgb_score(X_nb, y_nb, num_nb, cat_nb):.4f}")
print(f"  + Interaction features   : {get_lgb_score(X_int, y_int, num_int, cat_int):.4f}")
print("=" * 55)

   FEATURE EXPERIMENT RESULTS
  Baseline                 : 0.1279
  + Neighborhood encoding  : 0.1272
  + Interaction features   : 0.1293


# Submit best version to Kaggle:

In [7]:
# build best dataset — baseline + neighborhood encoding only
def prepare_data(df):
    df = engineer_features(df)
    df['Neighborhood_encoded'] = df['Neighborhood'].map(neighborhood_mean)
    df['Neighborhood_encoded'].fillna(neighborhood_mean.mean(), inplace=True)
    df = df.drop(columns=['Neighborhood'])
    return df

train_best = prepare_data(train)
test_best  = prepare_data(test)

X_best = train_best.drop(columns=['SalePrice', 'Id'])
y_best = np.log1p(train_best['SalePrice'])
X_test_best = test_best.drop(columns=['Id'])

num_best = X_best.select_dtypes(include=['int64','float64']).columns.tolist()
cat_best = X_best.select_dtypes(include=['object']).columns.tolist()

# final pipeline
num_tr = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                          ('scaler', StandardScaler())])
cat_tr = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                          ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                     unknown_value=-1))])
preprocessor_best = ColumnTransformer(transformers=[
    ('num', num_tr, num_best),
    ('cat', cat_tr, cat_best)
])

final_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor_best),
    ('model', lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.005,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_samples=20,
        reg_alpha=0.5,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1
    ))
])

# fit and predict
final_pipe.fit(X_best, y_best)
predictions = np.expm1(final_pipe.predict(X_test_best))

submission = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': predictions
})
submission.to_csv('lgb_best_submission.csv', index=False)
print("Submission saved!")
print(f"Prediction range: ${predictions.min():,.0f} to ${predictions.max():,.0f}")

Submission saved!
Prediction range: $52,170 to $499,502
